# **Section - 1**

## **Generating synthetic Data**

In [12]:
import sqlite3
import random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)


In [13]:

N_USERS        = 10000
N_SESSIONS     = 50000
N_TRANSACTIONS = 28000
N_VIEWS        = 30000


START_DATE = datetime(2024, 10, 1)
END_DATE   = datetime(2025, 1, 31)

COUNTRIES = {
    "IN": 0.45,
    "US": 0.20,
    "GB": 0.08,
    "AU": 0.05,
    "CA": 0.05,
    "DE": 0.04,
    "SG": 0.04,
    "AE": 0.04,
    "OTHER": 0.05
}

PLATFORMS = ["android", "ios", "web"]

PLAN_TYPES = ["free", "basic", "premium"]

GENRES = [
    "romance",
    "action",
    "thriller",
    "comedy",
    "horror",
    "sci-fi",
    "slice_of_life"
]

CONTENT_TITLES = [
    "Midnight Vows",
    "Steel Hearts",
    "Neon Blade",
    "Campus Crush",
    "Dark Whispers",
    "Galaxy Drift",
    "Sweet Revenge",
    "The Last Heir",
    "Blood Pact",
    "Cherry Blossom Trap",
    "Rogue Protocol",
    "Lovesick CEO",
    "Shadow Hunter",
    "My Demon Boyfriend",
    "Office War",
    "Electric Soul",
    "Crimson Thread",
    "Star-Crossed",
    "The Fake Marriage",
    "Underworld Kiss"
]

CONTENT_IDS = {
    title: i + 1
    for i, title in enumerate(CONTENT_TITLES)
}

TITLE_GENRE = {
    title: GENRES[i % len(GENRES)]
    for i, title in enumerate(CONTENT_TITLES)
}



### **Defining Users Table**


In [14]:
def rand_date():
    time_delta = END_DATE - START_DATE
    random_days = random.uniform(0, time_delta.days)
    random_seconds = random.uniform(0, 24 * 3600)
    return START_DATE + timedelta(days=random_days, seconds=random_seconds)

def weighted_choice(choices):
    items = list(choices.keys())
    weights = list(choices.values())
    return random.choices(items, weights=weights, k=1)[0]

users_rows = []

for uid in range(1, N_USERS + 1):

    signup = rand_date()

    plan = weighted_choice({
        "free": 0.55,
        "basic": 0.25,
        "premium": 0.20
    })

    users_rows.append({
        "user_id": uid,

        "country": weighted_choice(COUNTRIES),

        "platform": random.choice(PLATFORMS),

        "plan_type": plan,

        "signup_date": signup.date().isoformat(),

        "age_bucket": random.choice([
            "13-17",
            "18-24",
            "25-34",
            "35-44",
            "45+"
        ]),

        "gender": random.choice([
            "M",
            "F",
            "other"
        ]),

        "is_active": int(random.random() > 0.15)
    })

users_df = pd.DataFrame(users_rows)

print(f"✅ Users table: {len(users_df):,} rows")

users_df.head()

✅ Users table: 10,000 rows


,user_id,country,platform,plan_type,signup_date,age_bucket,gender,is_active
0,1,IN,web,free,2024-12-18,13-17,other,1
1,2,IN,android,free,2024-12-07,45+,other,0
2,3,IN,web,free,2024-10-25,25-34,M,1
3,4,IN,ios,free,2024-10-20,13-17,M,1
4,5,IN,ios,free,2024-11-14,45+,M,1


### **Defining Sessions Table**

In [15]:
sessions_rows = []

for sid in range(1, N_SESSIONS + 1):

    user = users_df.sample(1).iloc[0]

    start = rand_date()

    duration = max(
        30,
        int(np.random.exponential(600))
    )

    sessions_rows.append({
        "session_id": sid,

        "user_id": int(user["user_id"]),

        "platform": user["platform"],

        "session_start": start.isoformat(),

        "session_end": (
            start + timedelta(seconds=duration)
        ).isoformat(),

        "duration_seconds": duration,

        "country": user["country"]
    })

sessions_df = pd.DataFrame(sessions_rows)

print(f"✅ Sessions table: {len(sessions_df):,} rows")

sessions_df.head()

✅ Sessions table: 50,000 rows


,session_id,user_id,platform,session_start,session_end,duration_seconds,country
0,1,6253,android,2024-11-04T11:21:35.100378,2024-11-04T11:36:42.100378,907,IN
1,2,4615,android,2024-12-07T00:27:12.783430,2024-12-07T00:41:30.783430,858,CA
2,3,9914,ios,2024-12-11T17:54:37.105954,2024-12-11T18:03:37.105954,540,CA
3,4,4450,android,2024-12-14T15:14:20.730091,2024-12-14T15:34:38.730091,1218,IN
4,5,9230,android,2024-12-04T01:32:28.075453,2024-12-04T01:34:23.075453,115,IN


### **Defining Transaction Table**

In [17]:
transactions_rows = []

paying_users = users_df[
    users_df["plan_type"] != "free"
]["user_id"].tolist()

for tid in range(1, N_TRANSACTIONS + 1):

    uid = random.choice(paying_users)

    user_row = users_df[
        users_df["user_id"] == uid
    ].iloc[0]

    ts = rand_date()

    plan = user_row["plan_type"]

    amount = 4.99 if plan == "basic" else 9.99

    status = random.choices(
        ["success", "failed", "refunded"],
        weights=[0.90, 0.06, 0.04]
    )[0]

    transactions_rows.append({
        "transaction_id": tid,

        "user_id": int(uid),

        "timestamp": ts.isoformat(),

        "amount_usd": amount if status == "success" else 0,

        "plan_type": plan,

        "status": status,

        "payment_method": random.choice([
            "card",
            "upi",
            "wallet",
            "bank_transfer"
        ]),

        "country": user_row["country"]
    })

transactions_df = pd.DataFrame(transactions_rows)

print(f"✅ Transactions table: {len(transactions_df):,} rows")

transactions_df.head()

✅ Transactions table: 28,000 rows


,transaction_id,user_id,timestamp,amount_usd,plan_type,status,payment_method,country
0,1,640,2024-11-09T01:10:04.663352,4.99,basic,success,wallet,IN
1,2,1106,2024-12-10T10:34:48.077036,0.00,basic,failed,upi,IN
2,3,7150,2025-01-06T02:08:16.103092,4.99,basic,success,wallet,CA
3,4,7769,2024-11-06T14:57:52.635506,4.99,basic,success,card,IN
4,5,424,2024-10-24T07:35:07.254888,4.99,basic,success,upi,GB


### **Defining Content Table**

In [18]:
views_rows = []

for vid in range(1, N_VIEWS + 1):

    user = users_df.sample(1).iloc[0]

    title = random.choice(CONTENT_TITLES)

    ts = rand_date()

    views_rows.append({
        "view_id": vid,

        "user_id": int(user["user_id"]),

        "content_id": CONTENT_IDS[title],

        "content_title": title,

        "genre": TITLE_GENRE[title],

        "episode_number": random.randint(1, 30),

        "viewed_at": ts.isoformat(),

        "completion_pct": round(
            random.uniform(0.05, 1.0),
            2
        ),

        "is_ppv": int(random.random() < 0.30),

        "country": user["country"],

        "platform": user["platform"]
    })

views_df = pd.DataFrame(views_rows)

print(f"✅ Content views table: {len(views_df):,} rows")

views_df.head()

✅ Content views table: 30,000 rows


,view_id,user_id,content_id,content_title,genre,episode_number,viewed_at,completion_pct,is_ppv,country,platform
0,1,8399,7,Sweet Revenge,slice_of_life,29,2024-11-08T17:04:26.234698,0.40,0,US,ios
1,2,1244,12,Lovesick CEO,horror,24,2024-10-19T12:31:08.529766,0.25,0,SG,android
2,3,4751,6,Galaxy Drift,sci-fi,9,2025-01-23T11:12:11.380556,0.83,0,IN,web
3,4,2191,8,The Last Heir,romance,5,2024-12-14T11:13:55.196527,0.89,0,IN,web
4,5,3107,19,The Fake Marriage,horror,30,2024-12-11T02:42:22.817823,0.07,0,IN,ios


# **SECTION - 2**

### **Saving to Database**

In [19]:
DB_PATH = "pockettoons.db"

conn = sqlite3.connect(DB_PATH)

users_df.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

sessions_df.to_sql(
    "sessions",
    conn,
    if_exists="replace",
    index=False
)

transactions_df.to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

views_df.to_sql(
    "content_views",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print(f"✅ Database saved → {DB_PATH}")

✅ Database saved → pockettoons.db


In [20]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables in the database:")
for table in tables:
    print(f"- {table[0]}")

conn.close()


Tables in the database:
- users
- sessions
- transactions
- content_views


In [21]:
conn = sqlite3.connect(DB_PATH)

for table in [
    "users",
    "sessions",
    "transactions",
    "content_views"
]:

    count = pd.read_sql(
        f"SELECT COUNT(*) as rows FROM {table}",
        conn
    ).iloc[0, 0]

    print(f"{table:<20} {count:>10,} rows")

conn.close()

print("\n✅ Database verified!")

users                    10,000 rows
sessions                 50,000 rows
transactions             28,000 rows
content_views            30,000 rows

✅ Database verified!


In [22]:
conn = sqlite3.connect(DB_PATH)

tables_to_check = ["sessions", "transactions", "content_views"]

for table_name in tables_to_check:
    query = f"""
SELECT *
FROM {table_name}
LIMIT 5;
"""

    print(f"\n--- First 5 rows of {table_name} ---")
    df_first_5_rows = pd.read_sql(query, conn)
    print(df_first_5_rows.to_string())

conn.close()


--- First 5 rows of sessions ---
   session_id  user_id platform               session_start                 session_end  duration_seconds country
0           1     6253  android  2024-11-04T11:21:35.100378  2024-11-04T11:36:42.100378               907      IN
1           2     4615  android  2024-12-07T00:27:12.783430  2024-12-07T00:41:30.783430               858      CA
2           3     9914      ios  2024-12-11T17:54:37.105954  2024-12-11T18:03:37.105954               540      CA
3           4     4450  android  2024-12-14T15:14:20.730091  2024-12-14T15:34:38.730091              1218      IN
4           5     9230  android  2024-12-04T01:32:28.075453  2024-12-04T01:34:23.075453               115      IN

--- First 5 rows of transactions ---
   transaction_id  user_id                   timestamp  amount_usd plan_type   status payment_method country
0               1      640  2024-11-09T01:10:04.663352        4.99     basic  success         wallet      IN
1               2     1106

In [23]:
from google.colab import files
files.download("pockettoons.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>